In [1]:
import matplotlib.pyplot as plt
import numpy as np
import xpart as xp
import xtrack as xt
import yaml
from scipy.constants import c
from scipy.optimize import curve_fit
#%reload_ext autotime
%config InlineBackend.figure_format = "retina"

/afs/cern.ch/work/a/aradosla/private/example_DA_study_mine/miniforge/lib/python3.10/site-packages/cupyx/jit/_interface.py:173: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


In [2]:
collider = xt.Multiline.from_json("collider_final.json")
line = collider.lhcb1

FileNotFoundError: [Errno 2] No such file or directory: 'collider_final.json'

In [ ]:
particle_ref = xp.Particles(
                        mass0=xp.PROTON_MASS_EV, q0=1, energy0=450e9) #config_mad['beam_config']['lhcb1']['beam_energy_tot'])
bunch_intensity = 5e15
nemitt_x = 9.60530822375217e-07 #1e-6  # in [m]
nemitt_y = 0.6e-7  # in [m]
sigma_z = 7.5e-2  # bunch length in [m]                        

collider.vars['i_oct_b1'] =0
collider.vars['i_oct_b2'] =0

line.build_tracker()

gaussian_bunch = xp.generate_matched_gaussian_bunch(
            num_particles = 10000, total_intensity_particles = bunch_intensity,
            nemitt_x = nemitt_x, nemitt_y=nemitt_y, sigma_z = sigma_z,
            particle_ref = particle_ref,
            line = collider['lhcb1'])

In [ ]:
collider.vars['vrf400']._value

In [ ]:
ibs_kick = xf.IBSKineticKick(num_slices=5)
collider['lhcb1'].configure_intrabeam_scattering(
element=ibs_kick, name="ibskick", index=-1, update_every=5
)

In [ ]:
collider['lhcb1'].track(gaussian_bunch, num_turns=100, turn_by_turn_monitor=True, with_progress=5)

In [ ]:
x_data = line.record_last_track.x.T
y_data = line.record_last_track.y.T
px_data = line.record_last_track.px.T
py_data = line.record_last_track.py.T
zeta_data = line.record_last_track.zeta.T
pzeta_data = line.record_last_track.delta.T
turns = line.record_last_track.at_turn.T
particle_ids = line.record_last_track.particle_id.T
alldata = line.record_last_track

In [ ]:
import pandas as pd
dff = pd.DataFrame({'x': x_data.flatten(), 'y': y_data.flatten(), 'px': px_data.flatten(), 'py': py_data.flatten(), 'zeta':zeta_data.flatten(), 'pzeta': pzeta_data.flatten(), 'at_turn':turns.flatten(), 'particle_id':particle_ids.flatten() })
tw0 = line.twiss()
gamma_rel = gaussian_bunch.gamma0
betx_rel = gaussian_bunch.beta0
sigmax_col = np.sqrt(3.5e-6 / gamma_rel * tw0.betx[0])

In [ ]:

state_from0 = []
state0_where = []

for i in dff[dff.at_turn == 999].x:
    #x_data_filtered = np.delete(dff[dff.at_turn == turn].x, state_mine)
    where = np.where(i == 0)
    print(where)
    state0_where.append(where)
    state_from0.append(len(where[0]))
    

In [ ]:
#x_emittance
# Gaussian x
dff[dff.at_turn == 0].x

beam_size = nemitt_x/gamma_rel
def collimator(x_data_turn, px_data_turn):
    #survived = 1/2*(gamx*np.sum(x_data_turn)**2 + 2*alfx*np.sum(x_data_turn)*np.sum(px_data_turn) + betx*np.sum(px_data_turn)**2)
    survived = 1/2*(x_data_turn**2 + px_data_turn**2)
    #print(survived)
    return survived 
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean)**2 / (2 * sigma**2)))

gaussian_emit_all = []

for turn in range(100):
    #print(turn)
   
    x_data_filtered = np.isnan(dff[dff.at_turn == turn].x.dropna())
    #print(dff)
    #plt.hist(data_turn[collimator(data_turn.x_phys, data_turn.px_phys) < (6*sigmax_col)**2].x_phys/np.sqrt(tw0.betx[0]*beam_size), bins=100, density = True)
    hist, bin_edges = np.histogram(x_data_filtered/np.sqrt(tw0.betx[0]*beam_size[0]), bins=100, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Initial guess for the parameters
    initial_guess = [1, 1.0, 1]

    # Fit the q-Gaussian to the histogram data
    params, covariance = curve_fit(gaussian, bin_centers, hist, p0=initial_guess, maxfev = 2000000)
    
  
    # Plotting the q-Gaussian fit
    x = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    
    y = gaussian(x, *params)
    
    A, mean, sigma = params
       
    gaussian_sigma = sigma
    
   
    #print(data_turn)
    sigma_delta = float(np.std(dff[dff.at_turn==turn].pzeta/np.sqrt(tw0.betx[0]*beam_size[0])))
   
    gaussian_emit_geom = (gaussian_sigma**2-(tw0["dx"][0]*sigma_delta)**2)/tw0.betx[0]*tw0.betx[0]*beam_size[0]
    gaussian_emit = gaussian_emit_geom*(gamma_rel*betx_rel)

    gaussian_emit_all.append(gaussian_emit[0])
#plt.plot(gaussian_emit_all)
#plt.plot(qgaussian_q_all)

In [ ]:
#y_emittance
# Gaussian x

gaussian_sigma_all = []
gaussian_emit_all = []
beam_size = nemitt_x/gamma_rel
def collimator(x_data_turn, px_data_turn):
    #survived = 1/2*(gamx*np.sum(x_data_turn)**2 + 2*alfx*np.sum(x_data_turn)*np.sum(px_data_turn) + betx*np.sum(px_data_turn)**2)
    survived = 1/2*(x_data_turn**2 + px_data_turn**2)
    #print(survived)
    return survived 
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean)**2 / (2 * sigma**2)))

gaussian_emit_all_y = []
bunch_length = []
sigma_delta_all = []

for turn in range(100):
    #print(turn)
    
    try:
        state_mine = state0_where[turn][0]
        x_data_filtered = np.delete(dff[dff.at_turn == turn].x, state_mine)
        pzeta_filtered = np.delete(dff[dff.at_turn == turn].pzeta, state_mine)
        zeta_filtered = np.delete(dff[dff.at_turn == turn].zeta, state_mine)
        #plt.hist(y_data_filtered, bins = 100)
        #plt.title('f{turn}')
        #plt.show()
    except:
        x_data_filtered = dff[dff.at_turn == turn].x
        pzeta_filtered =dff[dff.at_turn == turn].pzeta
        zeta_filtered = dff[dff.at_turn == turn].zeta
   
    #plt.hist(data_turn[collimator(data_turn.x_phys, data_turn.px_phys) < (6*sigmax_col)**2].x_phys/np.sqrt(tw0.betx[0]*beam_size), bins=100, density = True)
    
    hist, bin_edges = np.histogram(x_data_filtered/np.sqrt(tw0.betx[0]*beam_size[0]), bins=100, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Initial guess for the parameters
    initial_guess = [1, 1.0, 1]

    # Fit the q-Gaussian to the histogram data
    params, covariance = curve_fit(gaussian, bin_centers, hist, p0=initial_guess, maxfev = 2000000)
    
  
    # Plotting the q-Gaussian fit
    x = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    
    y = gaussian(x, *params)
    
    A, mean, sigma = params
       
    gaussian_sigma = sigma
    
    #plt.hist(y_data_filtered/np.sqrt(tw0.bety[0]*beam_size[0]), bins=100, density=True)
    #plt.plot(x, y)
    #plt.show()
    #print(data_turn)
    sigma_delta = float(np.std(pzeta_filtered/np.sqrt(tw0.betx[0]*beam_size[0])))
    sigma_delta_all.append(sigma_delta*np.sqrt(tw0.betx[0]*beam_size[0]))
    bunch_length.append(np.std(zeta_filtered))
    gaussian_sigma_all.append(gaussian_sigma)
    gaussian_emit_geom = (gaussian_sigma**2-(tw0["dx"][0]*sigma_delta)**2)/tw0["betx"][0]*tw0.betx[0]*beam_size[0]
    print(gaussian_sigma**2-(tw0["dx"][0]*sigma_delta)**2)
    gaussian_emit = gaussian_emit_geom*(gamma_rel*betx_rel)
    
    gaussian_emit_all.append(gaussian_emit[0])
#plt.plot(gaussian_emit_all)
#plt.plot(qgaussian_q_all)

In [ ]:
plt.plot(gaussian_emit_all)

In [ ]:
#y_emittance
# Gaussian y
intensity_left = []
gaussian_sigma_all = []
beam_size = nemitt_y/gamma_rel
def collimator(x_data_turn, px_data_turn):
    #survived = 1/2*(gamx*np.sum(x_data_turn)**2 + 2*alfx*np.sum(x_data_turn)*np.sum(px_data_turn) + betx*np.sum(px_data_turn)**2)
    survived = 1/2*(x_data_turn**2 + px_data_turn**2)
    #print(survived)
    return survived 
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean)**2 / (2 * sigma**2)))

gaussian_emit_all_y = []
bunch_length = []
sigma_delta_all = []

for turn in range(100):
    #print(turn)
    
    try:
        state_mine = state0_where[turn][0]
        y_data_filtered = np.delete(dff[dff.at_turn == turn].y, state_mine)
        pzeta_filtered = np.delete(dff[dff.at_turn == turn].pzeta, state_mine)
        zeta_filtered = np.delete(dff[dff.at_turn == turn].zeta, state_mine)
        #plt.hist(y_data_filtered, bins = 100)
        #plt.title('f{turn}')
        #plt.show()
    except:
        y_data_filtered = dff[dff.at_turn == turn].y
        pzeta_filtered =dff[dff.at_turn == turn].pzeta
        zeta_filtered = dff[dff.at_turn == turn].zeta
   
    #plt.hist(data_turn[collimator(data_turn.x_phys, data_turn.px_phys) < (6*sigmax_col)**2].x_phys/np.sqrt(tw0.betx[0]*beam_size), bins=100, density = True)
    
    hist, bin_edges = np.histogram(y_data_filtered/np.sqrt(tw0.bety[0]*beam_size[0]), bins=100, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Initial guess for the parameters
    initial_guess = [1, 1.0, 1]

    # Fit the q-Gaussian to the histogram data
    params, covariance = curve_fit(gaussian, bin_centers, hist, p0=initial_guess, maxfev = 2000000)
    
  
    # Plotting the q-Gaussian fit
    x = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    
    y = gaussian(x, *params)
    
    A, mean, sigma = params
       
    gaussian_sigma = sigma
    
    #plt.hist(y_data_filtered/np.sqrt(tw0.bety[0]*beam_size[0]), bins=100, density=True)
    #plt.plot(x, y)
    #plt.show()
    #print(data_turn)
    sigma_delta = float(np.std(pzeta_filtered/np.sqrt(tw0.betx[0]*beam_size[0])))
    sigma_delta_all.append(sigma_delta*np.sqrt(tw0.betx[0]*beam_size[0]))
    bunch_length.append(np.std(zeta_filtered))
    gaussian_sigma_all.append(gaussian_sigma)
    gaussian_emit_geom = (gaussian_sigma**2-(tw0["dy"][0]*sigma_delta)**2)/tw0["bety"][0]*tw0.bety[0]*beam_size[0]
    print(gaussian_sigma**2-(tw0["dx"][0]*sigma_delta)**2)
    gaussian_emit = gaussian_emit_geom*(gamma_rel*betx_rel)
    gaussian_emit_all_y.append(gaussian_emit[0])
#plt.plot(gaussian_emit_all)
#plt.plot(qgaussian_q_all)

In [ ]:
plt.plot(gaussian_emit_all_y)